# Final model - multi-modal EfficientNet

Trains the final architecture across folds and saves weights, per-fold metrics
and out-of-fold predictions.

## Before you run

1. **Add Data** on the right: `melanoma-512-preprocessed`
2. **Accelerator: GPU T4 x2**. Not P100 - Kaggle's PyTorch dropped Pascal, and
   a P100 reports `cuda True` then fails on the first real operation.
3. **Internet: On** (timm downloads the pretrained backbone)
4. Set `FOLDS` in the next cell. Kaggle allows two concurrent GPU sessions:
   run `"0,1,2"` in one notebook and `"3,4"` in another and the wall clock
   roughly halves.

Everything is resumable. If the session dies, open it and Run All; finished
folds are skipped.


In [ ]:
# ============================================================
# THE ONLY LINE THAT DIFFERS BETWEEN YOUR TWO SESSIONS
FOLDS = "0,1,2"        # session A. Use "3,4" in session B.
# ============================================================

BACKBONE   = "tf_efficientnet_b3"
IMAGE_SIZE = 300       # set this from the benchmark cell below
EPOCHS     = 12
BATCH_SIZE = 32
TTA        = 4         # 1 none, 2 horizontal flip, 4 all flips
BUDGET_H   = 10.0

RUN_BENCHMARK = True   # 2 minutes, tells you what actually fits


## 1. Environment


In [ ]:
import os, shutil, subprocess, time, glob
t0 = time.time()

import torch
assert torch.cuda.is_available(), "No GPU. Right panel > Accelerator > GPU T4 x2."

name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
arch = f"sm_{major}{minor}"

# torch.cuda.is_available() is not enough on Kaggle. A P100 is sm_60 and current
# PyTorch builds no longer compile for it: is_available() returns True and then
# every kernel launch fails. Check the architecture is actually supported.
assert arch in torch.cuda.get_arch_list(), (
    f"{name} is {arch}, which this PyTorch does not support "
    f"({torch.cuda.get_arch_list()}). Switch the accelerator to GPU T4 x2.")

print("torch", torch.__version__)
print("gpu  ", name, f"({arch})")

try:
    import albumentations as A
    A.Affine
    print("albumentations", A.__version__)
except Exception:
    subprocess.run(["pip", "install", "-q", "albumentations>=2.0"], check=False)
    import albumentations as A
    print("albumentations installed:", A.__version__)


## 2. Find the data and the source files

Kaggle has mounted datasets at `/kaggle/input/<slug>/` and at
`/kaggle/input/datasets/<user>/<slug>/`, and it auto-extracts `.tar` uploads,
which adds another level. So walk the tree rather than assuming a depth.

The pruning line matters: `train_512` holds 57,855 files on a network mount and
descending into it turns a two second search into several minutes.


In [ ]:
IMG_SRC = None
META    = None
py_files = []

for root, dirs, files in os.walk("/kaggle/input"):
    if IMG_SRC is None and "train_512" in dirs:
        IMG_SRC = os.path.join(root, "train_512")
    if META is None and "metadata_clean.csv" in files:
        META = os.path.join(root, "metadata_clean.csv")
    py_files += [os.path.join(root, f) for f in files if f.endswith(".py")]
    dirs[:] = [d for d in dirs if d not in ("train_512", "test_512")]

assert IMG_SRC, "train_512 not found. Add the dataset, then RESTART the session."
assert META, ("metadata_clean.csv not found. The plain folds.csv will not do, "
              "the model needs the sex/site/age columns.")

WORK = "/kaggle/working"
SRC  = os.path.join(WORK, "src")
shutil.rmtree(SRC, ignore_errors=True)   # start clean, no stale files from a previous run
os.makedirs(SRC, exist_ok=True)
for p in py_files:
    shutil.copy(p, SRC)

# train_final.py imports `models.final_model` and `losses.focal_loss`, so those
# have to be packages. Kaggle's uploader flattens subdirectories, so rebuild the
# layout here.
#
# models.py must not survive alongside a models/ package: Python resolves the
# module before the package, and `import models.final_model` then fails with
# "'models' is not a package".
for stale in ("models.py", "train.py"):
    stale_path = os.path.join(SRC, stale)
    if os.path.exists(stale_path):
        os.remove(stale_path)

for pkg, members in (("models", ("final_model.py", "baseline_models.py")),
                     ("losses", ("focal_loss.py",))):
    pkg_dir = os.path.join(SRC, pkg)
    os.makedirs(pkg_dir, exist_ok=True)
    open(os.path.join(pkg_dir, "__init__.py"), "w").close()
    for member in members:
        flat = os.path.join(SRC, member)
        if os.path.exists(flat):
            shutil.copy(flat, os.path.join(pkg_dir, member))

print("img_src :", IMG_SRC)
print("metadata:", META)
print("sources :", len(py_files), "files")

for need in ("run_final_kaggle.py", "final_report.py", "bench_backbones.py"):
    assert os.path.exists(os.path.join(SRC, need)), (
        f"{need} missing. Add the `melanoma-src` dataset as a second input, "
        "then RESTART the session.")
assert os.path.exists(os.path.join(SRC, "models", "final_model.py")), "final_model.py missing"
assert not os.path.exists(os.path.join(SRC, "models.py")), "models.py would shadow the package"

# Prove the imports actually resolve before spending hours on a run.
import subprocess
check = subprocess.run(
    ["python", "-c", "import models.final_model, losses.focal_loss, train_final; print('imports OK')"],
    cwd=SRC, capture_output=True, text=True)
print(check.stdout.strip() or check.stderr.strip()[-400:])
assert "imports OK" in check.stdout, "imports failed, see above"


## 3. Benchmark

Measures real images/sec on this GPU for five backbone and resolution pairs and
projects the hours a full run would take. Pick the largest row that says `yes`
or `2 sessions`, then set `BACKBONE` and `IMAGE_SIZE` at the top and rerun from
cell 1.

Do this **before** the resize cell. Redoing 57,855 images costs five minutes you
should not spend twice.


In [ ]:
if RUN_BENCHMARK:
    !cd {SRC} && python bench_backbones.py \
        --n_train 51228 --folds 5 --epochs {EPOCHS} \
        --batch_size {BATCH_SIZE} --budget_h {BUDGET_H}
else:
    print("skipped")


## 4. Resize the photos once

The stored photos are 512px. Decoding those and shrinking them every epoch makes
the four CPU workers the bottleneck and leaves the GPU idle about two thirds of
the time. Converting once to the training size fixes that.

`IMREAD_REDUCED_COLOR_2` decodes the JPEG straight to half size using DCT
scaling, which is far cheaper than a full decode followed by a resize.


In [ ]:
import cv2
from multiprocessing import Pool

FAST = f"/kaggle/working/train_{IMAGE_SIZE}"
os.makedirs(FAST, exist_ok=True)

names = [f for f in os.listdir(IMG_SRC) if f.endswith(".jpg")]
todo  = [n for n in names if not os.path.exists(os.path.join(FAST, n))]
print(f"{len(names)} photos, {len(todo)} to convert")

def shrink(n):
    img = cv2.imread(os.path.join(IMG_SRC, n), cv2.IMREAD_REDUCED_COLOR_2)
    if img is None:
        return n
    interp = cv2.INTER_AREA if img.shape[0] > IMAGE_SIZE else cv2.INTER_LINEAR
    img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE), interpolation=interp)
    ok, buf = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 92])
    if not ok:
        return n
    tmp = os.path.join(FAST, n) + ".tmp"
    with open(tmp, "wb") as h:
        h.write(buf.tobytes())
    os.rename(tmp, os.path.join(FAST, n))
    return None

if todo:
    with Pool(4) as pool:
        failed = [r for r in pool.imap_unordered(shrink, todo, chunksize=64) if r]
    print("failed:", len(failed))

print("ready:", len(os.listdir(FAST)), "photos in", FAST)
assert len(os.listdir(FAST)) == len(names)


## 5. Train

`--save_weights` is what produces the checkpoints. Do not drop it.

Each finished fold is written to `final_results.csv` immediately, so a dead
session costs one fold rather than the whole run.


In [ ]:
BUDGET = max(BUDGET_H - (time.time() - t0) / 3600, 0.5)
print(f"time budget: {BUDGET:.2f} h, folds {FOLDS}")

!cd {SRC} && python run_final_kaggle.py \
    --img_dir "{FAST}" \
    --metadata_csv "{META}" \
    --out_dir {WORK}/reports \
    --backbone {BACKBONE} \
    --image_size {IMAGE_SIZE} \
    --folds {FOLDS} \
    --epochs {EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --tta {TTA} \
    --num_workers 4 \
    --save_weights \
    --time_budget_h {BUDGET:.2f}


## 6. Report

Per fold, out of fold, the ensemble, and a direct comparison against the
resnet34 baseline (ROC-AUC 0.8873, PR-AUC 0.2285).

With only some folds in this session the out-of-fold numbers cover only those
folds. Merge both sessions' `final_results.csv` and `final_oof.csv` afterwards
and rerun this to get the real figures.


In [ ]:
!cd {SRC} && python final_report.py --in_dir {WORK}/reports

import pandas as pd
pd.read_csv(f"{WORK}/reports/final_results.csv")


## 7. Clean up, then Save Version

Delete only the resized photos. **Do not delete `{SRC}`** - if `--out_dir`
resolved relatively then `reports/` lives inside it, and removing it takes the
results with it.

Click **Save Version** afterwards so the outputs survive the session expiring.


In [ ]:
shutil.rmtree(FAST, ignore_errors=True)

total = 0
for root, dirs, files in os.walk(f"{WORK}/reports"):
    for f in sorted(files):
        p = os.path.join(root, f)
        size = os.path.getsize(p)
        total += size
        print(f"{size/1e6:9.2f} MB  {p}")
print(f"{'-'*40}\n{total/1e6:9.2f} MB  total")

weights = glob.glob(f"{WORK}/reports/weights/*.pth")
print(f"\ncheckpoints saved: {len(weights)}")
for w in sorted(weights):
    print("  ", os.path.basename(w))
